# Accented speech by prosody transfer: full text

The **target** voice provides the timbre and the segments. The **donor** voice (another language) provides the prosody: pitch, energy and/or duration.
This works on full texts, one sentence per chunk, like `synth_aligned`, and plays in the same interactive HTML player with a pitch/energy panel added.

**Normalization over the whole text** (`norm_scope="text"`, default): the mean and range of each voice are computed over *all* sentences.
Sentence-level differences of the donor (e.g. a question sitting higher, a sentence-final drop) are kept.
`norm_scope="sentence"` normalizes each sentence on its own, which removes those.

| feature | transferred | normalized to the target |
|---|---|---|
| pitch | donor's F0 contour | log-F0 mean (`"mean"`) or mean + range (`"meanstd"`) |
| energy | donor's N contour | mean or mean + range |
| duration | donor's per-token durations | `"rate"`: donor rhythm at the target's total speaking time; `"raw"`: donor tempo too |

In [2]:
from functions import *          # model, g2p, _encode, _voice_pack, _durations_cached, align_phonemes,
                                 # text_to_ipa_chunks, audiovisualize_interactive, manipulate, SR, FRAME_SAMPLES ...
import functions as F            # to reuse the player's HTML template
from functions import _encode, _voice_pack, _durations_cached   # underscore names are not exported by *
import torch, numpy as np, json, os, uuid
import soundfile as sf
from functions import * 
from functions import  _voice_pack, _encode,   predict_durations   , align_phonemes   # model, g2p, _encode, _voice_pack, predict_durations, align_phonemes, SR, FRAME_SAMPLES
import torch, numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Audio
from IPython.display import display, HTML, Audio

HOP   = FRAME_SAMPLES // 2       # F0 / energy frame: 300 samples = 12.5 ms
HOP_S = HOP / SR

c:\Users\gmoha\anaconda3\envs\kokoro\Lib\site-packages\torch\nn\modules\rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
c:\Users\gmoha\anaconda3\envs\kokoro\Lib\site-packages\torch\nn\utils\weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


## Model helpers

In [3]:
@torch.no_grad()
def make_alignment(ps, dur):
    """Token-to-frame alignment matrix (1, n_tokens, n_frames) from durations."""
    dev = model.device
    n = _encode(ps)["n"]
    dur = torch.as_tensor(dur, device=dev).long().clamp(min=0).reshape(-1)
    assert dur.shape[0] == n, f"durations ({dur.shape[0]}) != tokens incl. BOS/EOS ({n})"
    idx = torch.repeat_interleave(torch.arange(n, device=dev), dur)
    aln = torch.zeros((n, idx.shape[0]), device=dev)
    aln[idx, torch.arange(idx.shape[0], device=dev)] = 1
    return aln.unsqueeze(0)


@torch.no_grad()
def predict_f0n(ps, pack, aln):
    """Predicted F0 (Hz) and energy N of voice `pack` on the given alignment (numpy, 2*n_frames)."""
    enc = _encode(ps)
    s = pack[len(ps) - 1][:, 128:]
    d = model.predictor.text_encoder(enc["d_en"], s, enc["L"], enc["tm"])
    F0, N = model.predictor.F0Ntrain(d.transpose(-1, -2) @ aln, s)
    return F0.squeeze().cpu().numpy(), N.squeeze().cpu().numpy()


@torch.no_grad()
def decode(ps, pack, aln, f0, n):
    """Decoder with voice `pack`'s timbre and the GIVEN F0 / energy curves."""
    enc = _encode(ps)
    ref_s = pack[len(ps) - 1]
    to_t = lambda x: torch.as_tensor(x, dtype=torch.float32, device=model.device).unsqueeze(0)
    return model.decoder(enc["t_en"] @ aln, to_t(f0), to_t(n), ref_s[:, :128]).squeeze().cpu().numpy()

## Normalization helpers

In [4]:
VOICED_HZ = 50.0

def voiced_mask(f0, thr=VOICED_HZ, rel=0.6):
    """Voiced = above thr AND above rel * the median voiced pitch (drops on/offset frames)."""
    v = f0 > thr
    return v & (f0 > rel * np.median(f0[v])) if v.any() else v


def f0_stats(f0_list):
    """(mean, std) of log-F0 over the voiced frames of several chunks."""
    lf = np.log(np.concatenate([f[voiced_mask(f)] for f in f0_list]))
    return lf.mean(), lf.std()


def en_stats(n_list):
    x = np.concatenate(n_list)
    return x.mean(), x.std()


def apply_f0(f0_d, st_d, st_t, mode):
    """Donor contour into the target's register. Unvoiced frames keep the donor's values."""
    if mode == "raw":
        return f0_d.copy()
    out, v = f0_d.copy(), voiced_mask(f0_d)
    z = np.log(f0_d[v]) - st_d[0]
    if mode == "meanstd":
        z = z / st_d[1] * st_t[1]
    out[v] = np.exp(z + st_t[0])
    return out


def apply_energy(n_d, st_d, st_t, mode):
    if mode == "raw":
        return n_d.copy()
    z = n_d - st_d[0]
    if mode == "meanstd":
        z = z / (st_d[1] + 1e-8) * st_t[1]
    return z + st_t[0]


def scale_durations(dur, factor, total=None):
    """Multiply durations by `factor` and round so the sum is EXACTLY `total`
    (default: round(sum * factor)), with every token >= 1 frame.
    Largest-remainder rounding: frames go to / come from the tokens whose
    rounding error is largest, so each token stays as close as possible to dur*factor."""
    x = np.asarray(dur, float) * factor
    total = int(round(x.sum())) if total is None else int(total)
    assert total >= len(x), f"total {total} < number of tokens {len(x)}"
    out = np.maximum(np.floor(x), 1).astype(int)
    while out.sum() < total:                       # add to the most under-rounded tokens
        k = min(total - out.sum(), len(x))
        out[np.argsort(-(x - out))[:k]] += 1
    while out.sum() > total:                       # remove from the most over-rounded tokens (never below 1)
        cand = np.where(out > 1)[0]
        k = min(out.sum() - total, len(cand))
        out[cand[np.argsort(x[cand] - out[cand])[:k]]] -= 1
    return out.tolist()

## Full-text synthesis with prosody transfer
Same return values as `synth_aligned` (`audio, segments, seg_words, seg_altered`) plus `pros`:
the time-stamped curves for the player, and the per-sentence `(audio, info)` in `pros["chunks"]`,
which can be passed straight to `compare()` from the measures notebook.

In [5]:
def synth_prosody(ipa_chunks, target_voice, donor_voice,
                  pitch=True, energy=True, duration=False,
                  f0_mode="meanstd", energy_mode="meanstd", dur_mode="rate",
                  norm_scope="text", dur_scope="sentence", masks=None, speed=1.0, sr=SR):
    """dur_scope (dur_mode="rate" only):
         "sentence" -> every sentence gets exactly the target's number of frames
         "text"     -> only the whole text matches; sentences keep the donor's relative lengths"""
    assert norm_scope in ("text", "sentence") and dur_scope in ("text", "sentence") and dur_mode in ("raw", "rate")
    tpack, tkey = _voice_pack(target_voice)
    dpack, dkey = _voice_pack(donor_voice)

    # ---- pass 1: durations of both voices for every sentence ----
    items = []
    for ci, ps in enumerate(ipa_chunks):
        if not ps.strip():
            continue
        items.append(dict(ci=ci, ps=ps,
                          dur_t=_durations_cached(ps, tpack, tkey, speed),
                          dur_d=_durations_cached(ps, dpack, dkey, speed)))

    if duration and dur_mode == "rate" and dur_scope == "text":
        # whole-text frame budget split over sentences by cumulative rounding -> text total is exact
        f = sum(sum(it["dur_t"]) for it in items) / sum(sum(it["dur_d"]) for it in items)
        cum = np.round(np.cumsum([0] + [sum(it["dur_d"]) * f for it in items])).astype(int)
        budgets = np.diff(cum)
    for i, it in enumerate(items):
        if not duration:
            it["dur"] = list(it["dur_t"])
        elif dur_mode == "raw":
            it["dur"] = list(it["dur_d"])
        elif dur_scope == "sentence":   # donor rhythm, exactly the target's length per sentence
            it["dur"] = scale_durations(it["dur_d"], sum(it["dur_t"]) / sum(it["dur_d"]), total=sum(it["dur_t"]))
        else:
            it["dur"] = scale_durations(it["dur_d"], f, total=budgets[i])

    # ---- pass 2: F0 / energy of both voices on the SAME alignment ----
    for it in items:
        it["aln"] = make_alignment(it["ps"], it["dur"])
        it["f0_t"], it["n_t"] = predict_f0n(it["ps"], tpack, it["aln"])
        it["f0_d"], it["n_d"] = predict_f0n(it["ps"], dpack, it["aln"])

    # ---- pass 3: normalize (statistics over the whole text or per sentence) ----
    if norm_scope == "text":
        sf_t, sf_d = f0_stats([it["f0_t"] for it in items]), f0_stats([it["f0_d"] for it in items])
        se_t, se_d = en_stats([it["n_t"] for it in items]), en_stats([it["n_d"] for it in items])
    for it in items:
        if norm_scope == "sentence":
            sf_t, sf_d = f0_stats([it["f0_t"]]), f0_stats([it["f0_d"]])
            se_t, se_d = en_stats([it["n_t"]]), en_stats([it["n_d"]])
        it["f0"] = apply_f0(it["f0_d"], sf_d, sf_t, f0_mode)          if pitch  else it["f0_t"]
        it["n"]  = apply_energy(it["n_d"], se_d, se_t, energy_mode)   if energy else it["n_t"]

    # ---- pass 4: decode + segments, concatenated like synth_aligned ----
    audios, segments, seg_words, seg_altered = [], [], [], []
    T, F0o, F0t, Ne, Nt, chunks = [], [], [], [], [], []
    t0, word_offset = 0.0, 0
    for it in items:
        a = decode(it["ps"], tpack, it["aln"], it["f0"], it["n"])
        mask = masks[it["ci"]] if masks is not None else None
        segs, widx, altd = align_phonemes(it["ps"], it["dur"], n_samples=len(a), sr=sr, altered_mask=mask)
        segments += [(lab, s + t0, e + t0, info) for lab, s, e, info in segs]
        seg_words += [None if w is None else w + word_offset for w in widx]
        seg_altered += altd
        real = [w for w in widx if w is not None]
        word_offset += (max(real) + 1) if real else 0

        k = len(it["f0"])                                     # prosody curves on the global timeline
        T.append(t0 + np.arange(k) * (len(a) / k) / sr)
        F0o.append(np.where(voiced_mask(it["f0"]), it["f0"], np.nan))
        F0t.append(np.where(voiced_mask(it["f0_t"]), it["f0_t"], np.nan))
        Ne.append(it["n"]); Nt.append(it["n_t"])
        for arr, fill in [(T, t0 + len(a) / sr), (F0o, np.nan), (F0t, np.nan), (Ne, np.nan), (Nt, np.nan)]:
            arr.append(np.array([fill]))                      # NaN gap -> curves break between sentences

        info = {k_: it[k_] for k_ in ("ps", "dur", "dur_t", "dur_d", "f0", "f0_t", "f0_d", "n", "n_t", "n_d")}
        info.update(target=target_voice, donor=donor_voice,
                    what=[w for w, on in [("pitch", pitch), ("energy", energy), ("duration", duration)] if on])
        chunks.append((a, info))
        audios.append(a); t0 += len(a) / sr

    pros = dict(t=np.concatenate(T), f0=np.concatenate(F0o), f0_t=np.concatenate(F0t),
                n=np.concatenate(Ne), n_t=np.concatenate(Nt), chunks=chunks,
                target=target_voice, donor=donor_voice,
                what=[w for w, on in [("pitch", pitch), ("energy", energy), ("duration", duration)] if on],
                modes=dict(f0=f0_mode, energy=energy_mode, dur=dur_mode, scope=norm_scope, dur_scope=dur_scope))
    return np.concatenate(audios), segments, seg_words, seg_altered, pros

## Interactive player with a pitch / energy panel
Reuses your `audiovisualize_interactive` page (word boxes, token durations, waveform, click-to-play, wheel zoom).
Under the waveform there is a canvas that follows the same zoom and cursor, showing:
- **solid amber**: the F0 / energy that was actually synthesized
- **dashed grey**: what the target voice itself would have done on the same timing (its native prosody)

Every element id is made unique, so several players can sit in one notebook without controlling each other's audio.

In [6]:
_PROS_PANEL = r"""
<div id="prosLegend" style="font-size:11.5px;color:#555;margin:8px 0 2px;display:flex;gap:14px;flex-wrap:wrap;align-items:center;">
  <span><span style="display:inline-block;width:22px;border-top:2.5px solid #f59e0b;vertical-align:middle"></span> synthesized</span>
  <span><span style="display:inline-block;width:22px;border-top:2px dashed #94a3b8;vertical-align:middle"></span> target's own prosody</span>
  <span id="prosInfo" style="color:#777"></span>
</div>
<canvas id="pc" width="1040" height="150" style="width:100%;height:150px;background:#fff;border:1px solid #e5e7eb;border-radius:6px;cursor:pointer"></canvas>
"""

_PROS_JS = r"""
const PROS=__PROS__;
const pc=document.getElementById('pc'), pctx=pc.getContext('2d');
document.getElementById('prosInfo').textContent = PROS.info;
function drawPros(){
    const W=pc.width, H=pc.height, a=view[0], b=view[1];
    pctx.clearRect(0,0,W,H);
    const H1=Math.round(H*0.64);
    const lo=Math.log(PROS.f0lim[0]), hi=Math.log(PROS.f0lim[1]);
    const X=t=>(t-a)/(b-a)*W;
    const yF=f=>H1-6-(Math.log(f)-lo)/(hi-lo)*(H1-16);
    const yE=e=>H-4-(e-PROS.nlim[0])/(PROS.nlim[1]-PROS.nlim[0])*(H-H1-14);
    pctx.font='10px system-ui,sans-serif'; pctx.lineWidth=1;
    PROS.ticks.forEach(f=>{ const y=yF(f); pctx.strokeStyle='#f1f1f1';
        pctx.beginPath(); pctx.moveTo(0,y); pctx.lineTo(W,y); pctx.stroke();
        pctx.fillStyle='#999'; pctx.fillText(f+' Hz',3,y-2); });
    pctx.strokeStyle='#e5e7eb'; pctx.beginPath(); pctx.moveTo(0,H1+2); pctx.lineTo(W,H1+2); pctx.stroke();
    pctx.fillStyle='#999'; pctx.fillText('energy',3,H1+13);
    pctx.strokeStyle='#00000012';                                   // word starts, like the waveform
    groups.forEach(g=>{ if(g.w===null||g.s<a||g.s>b) return; const x=X(g.s);
        pctx.beginPath(); pctx.moveTo(x,0); pctx.lineTo(x,H); pctx.stroke(); });
    const t=PROS.t;
    function line(arr, yf, col, dash, lw){
        pctx.strokeStyle=col; pctx.setLineDash(dash); pctx.lineWidth=lw; pctx.beginPath();
        let pen=false;
        for(let i=0;i<t.length;i++){
            const v=arr[i];
            if(v===null || t[i]<a || t[i]>b){ pen=false; continue; }
            const x=X(t[i]), y=yf(v);
            if(pen) pctx.lineTo(x,y); else { pctx.moveTo(x,y); pen=true; }
        }
        pctx.stroke(); pctx.setLineDash([]); pctx.lineWidth=1;
    }
    line(PROS.f0_t, yF, '#94a3b8', [4,3], 1.5);
    line(PROS.f0,   yF, '#f59e0b', [], 2.2);
    line(PROS.n_t,  yE, '#94a3b8', [4,3], 1.2);
    line(PROS.n,    yE, '#f59e0b', [], 1.8);
    const cx=X(au.currentTime); pctx.strokeStyle='#f43f5e'; pctx.lineWidth=2;
    pctx.beginPath(); pctx.moveTo(cx,0); pctx.lineTo(cx,H); pctx.stroke(); pctx.lineWidth=1;
}
"""

_IDS = ["pv", "pageTitle", "rulesBox", "statsBox", "statsTotal", "statsBody", "au", "pp", "sp", "spl",
        "tt", "legend", "durNote", "wf", "ph", "pc", "prosLegend", "prosInfo"]


def _prosody_template():
    """The player template from functions.py with the prosody panel spliced in."""
    t = F._TEMPLATE
    anchors = ['<canvas id="wf"', "let view=[0,DUR];", "function drawWave(){", "\ndrawWave();\n})();"]
    missing = [a for a in anchors if a not in t]
    assert not missing, f"player template changed, anchors not found: {missing}"
    i = t.index('<canvas id="wf"'); j = t.index("</canvas>", i) + len("</canvas>")
    t = t[:j] + _PROS_PANEL + t[j:]
    t = t.replace("let view=[0,DUR];", "let view=[0,DUR];\n" + _PROS_JS, 1)
    t = t.replace("function drawWave(){", "function drawWave(){ drawPros();", 1)
    t = t.replace("\ndrawWave();\n})();", "\npc.onclick=wf.onclick; pc.onwheel=wf.onwheel;\ndrawWave();\n})();", 1)
    return t


def _uniquify_ids(html):
    uid = uuid.uuid4().hex[:8]
    for i in _IDS:
        html = html.replace(f'id="{i}"', f'id="{i}_{uid}"').replace(f"getElementById('{i}')", f"getElementById('{i}_{uid}')")
    return html


def _pros_payload(pros):
    tolist = lambda x, r: [None if not np.isfinite(v) else round(float(v), r) for v in x]
    f0_all = np.concatenate([pros["f0"], pros["f0_t"]]); f0_all = f0_all[np.isfinite(f0_all)]
    lo, hi = np.percentile(f0_all, [1, 99]); lo, hi = lo / 1.15, hi * 1.15
    n_all = np.concatenate([pros["n"], pros["n_t"]]); n_all = n_all[np.isfinite(n_all)]
    ticks = [f for f in (50, 75, 100, 150, 200, 300, 400, 500) if lo < f < hi]
    m = lambda f: np.exp(np.nanmean(np.log(f)))
    what = " + ".join(pros["what"]) or "nothing"
    md = pros["modes"]
    info = (f"{what} from {pros['donor']}"
            + (f"  ·  F0 {md['f0']}, energy {md['energy']}, duration {md['dur']}, stats per {md['scope']}" if pros["what"] else "")
            + f"  ·  mean F0 {m(pros['f0']):.0f} Hz (target's own {m(pros['f0_t']):.0f} Hz)")
    return dict(t=tolist(pros["t"], 4), f0=tolist(pros["f0"], 1), f0_t=tolist(pros["f0_t"], 1),
                n=tolist(pros["n"], 3), n_t=tolist(pros["n_t"], 3),
                f0lim=[float(lo), float(hi)], nlim=[float(n_all.min()), float(n_all.max())],
                ticks=ticks, info=info)


def audiovisualize_prosody(audio, segments, pros, sr=SR, out_html=None, seg_words=None, text=None,
                           seg_altered=None, title=None, rules=None, stats=None):
    """audiovisualize_interactive + pitch/energy panel. Same arguments, plus `pros` from synth_prosody."""
    original = F._TEMPLATE
    try:
        F._TEMPLATE = _prosody_template()
        html = audiovisualize_interactive(audio, segments, sr=sr, out_html=None, seg_words=seg_words,
                                          text=text, seg_altered=seg_altered, title=title,
                                          rules=rules, stats=stats)
    finally:
        F._TEMPLATE = original
    html = _uniquify_ids(html.replace("__PROS__", json.dumps(_pros_payload(pros))))
    if out_html:
        out_dir = os.path.join(os.getcwd(), "output_online"); os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, out_html)
        page = ('<!DOCTYPE html>\n<html><head><meta charset="utf-8">\n'
                '<meta name="viewport" content="width=device-width, initial-scale=1">\n'
                f'<title>{title or "Kokoro"}</title></head>\n<body>\n' + html + '\n</body></html>\n')
        with open(path, "w", encoding="utf-8") as f:
            f.write(page)
        print(f"Writing interactive HTML to: {path}")
    return html

## Settings

In [7]:
text = (""" Elena Marchetti lavorava da sei anni nell'archivio comunale di Ponteverde, una piccola città costruita lungo un fiume nella pianura del Nord. L'archivio occupava il terzo piano di un vecchio palazzo di mattoni rossi, a pochi passi dalla piazza principale e dal municipio. Il palazzo era stato costruito alla fine dell'Ottocento, e in passato aveva ospitato una scuola e poi alcuni uffici del tribunale. La sala più grande aveva soffitti alti, scaffali di legno scuro e due lunghe finestre che guardavano verso il fiume. In quella sala si trovavano i registri delle nascite, dei matrimoni e delle proprietà, alcuni dei quali avevano più di duecento anni. Il lavoro di Elena consisteva nel catalogare i documenti, nel rispondere alle richieste dei cittadini e nel tenere in ordine le sale. Ogni mattina arrivava alle otto, accendeva le luci, apriva le finestre per qualche minuto e controllava la posta elettronica dell'ufficio. Di solito restava in archivio fino alle sette di sera, perché preferiva lavorare quando il palazzo era tranquillo e silenzioso. Nelle ore del pomeriggio gli altri uffici si svuotavano, e lei poteva concentrarsi sui registri più vecchi senza essere interrotta.
Con lei lavorava anche Paolo Ferretti, un collega più giovane che si occupava soprattutto della digitalizzazione dei documenti. Paolo passava molte ore davanti allo scanner, una grande macchina che fotografava le pagine una per una con molta precisione. Quando aveva tempo, Paolo aiutava Elena anche con le richieste dei cittadini, soprattutto quelle che arrivavano per posta elettronica. Al piano terra, invece, c'era il custode del palazzo, Aldo Rinaldi, che lavorava in quell'edificio da più di trent'anni. Aldo conosceva ogni stanza, ogni porta e ogni scala, e spesso raccontava come era cambiato il palazzo nel corso del tempo. Ogni giovedì Elena passava qualche minuto con Aldo nell'ingresso, per parlare del tempo, dei lavori in città e delle novità del comune.
All'inizio di novembre, una sera, Elena ha sentito per la prima volta un rumore strano che veniva dal soffitto della sala grande. Era una serie di piccoli colpi secchi, simili al suono di qualcuno che batte lentamente con le nocche su una porta di legno. Non assomigliava ai rumori normali di un edificio vecchio, come il vento nelle finestre o i passi delle persone nelle scale. Elena si è alzata dalla scrivania, ha camminato fino al centro della sala e ha guardato il soffitto per qualche momento. Ha pensato per un attimo a una finestra aperta, ma tutte le finestre della sala erano chiuse e il vento non era forte. Non ha visto niente di strano, perché l'intonaco era bianco e liscio come sempre, e le lampade erano ferme al loro posto. Il rumore è durato circa dieci minuti, poi si è fermato all'improvviso, e la sala è tornata silenziosa come sempre. Elena ha pensato che qualcuno stesse lavorando al piano di sopra, ma poi si è ricordata che il terzo piano era l'ultimo. Sopra l'archivio c'era soltanto il tetto, con le travi di legno e uno spazio basso dove nessuno entrava da molti anni.
La sera seguente, verso la stessa ora, il rumore è ricominciato, sempre dallo stesso punto del soffitto, vicino alla finestra. Anche questa volta è durato pochi minuti, e poi è scomparso senza lasciare nessuna traccia e nessuna spiegazione evidente. Elena ha preso un foglio e ha scritto la data, l'ora e una breve descrizione del suono, per non dimenticare i dettagli. Nei giorni successivi il rumore si è ripetuto quasi ogni sera, e qualche volta Elena lo ha sentito anche al mattino presto. Non era un suono forte, ma in una sala così silenziosa era impossibile non notarlo e non chiedersi da dove venisse. Per abitudine, Elena ha messo il foglio nel primo cassetto della scrivania, insieme alle chiavi e agli altri appunti della settimana.
Una mattina Elena ha raccontato tutto a Paolo, mentre preparavano insieme le scatole di documenti da portare allo scanner. Paolo l'ha ascoltata con attenzione e poi ha detto che, secondo lui, nel soffitto potevano esserci dei topi o altri piccoli animali. Ha spiegato che negli edifici vecchi succede spesso, soprattutto in autunno, quando gli animali cercano un posto caldo per l'inverno. Ha raccontato che anche nella casa dei suoi nonni, in campagna, una volta avevano trovato dei topi sotto il tetto. In quel caso, ha detto, gli animali facevano rumore soprattutto di notte, quando la casa era buia e tranquilla. "Se ci sono dei topi sopra la sala, dobbiamo chiamare subito qualcuno, perché potrebbero rovinare anche i documenti", ha detto Paolo. Elena ha pensato che l'idea fosse ragionevole, perché i colpi erano irregolari e potevano sembrare i movimenti di un animale. Tuttavia, le sembrava strano che gli animali facessero rumore sempre alla stessa ora, e quasi sempre per lo stesso numero di minuti. Inoltre, durante il giorno la sala era piena di rumori normali, e forse gli animali si muovevano anche in quelle ore senza farsi sentire. Ha deciso comunque di seguire il consiglio del collega, perché la protezione dei documenti era la cosa più importante per l'archivio. Nel pomeriggio ha scritto una richiesta all'ufficio tecnico del comune, chiedendo un controllo del soffitto e dello spazio sotto il tetto.
Pochi giorni dopo è arrivato un tecnico di una ditta specializzata, con una scala, una torcia potente e una piccola telecamera. La ditta lavorava spesso per il comune, e controllava regolarmente le scuole, le palestre e gli altri edifici pubblici della città. Il tecnico è salito nello spazio sotto il tetto attraverso una botola nel corridoio, che Elena non aveva mai notato prima. Elena e Paolo hanno aspettato nel corridoio, ascoltando i passi del tecnico che si muoveva lentamente sopra le loro teste. Ogni tanto sentivano anche la sua voce, perché parlava al telefono con un collega per descrivere quello che vedeva. Ha passato quasi un'ora a controllare le travi, gli angoli e il pavimento di legno sopra la sala grande dell'archivio. Ha usato la telecamera per guardare anche nei punti più stretti, dove una persona non riusciva a entrare con il corpo. Le immagini apparivano su un piccolo schermo, e il tecnico le osservava con calma prima di passare al punto successivo. Quando è sceso, ha spiegato a Elena e a Paolo che non aveva trovato nessun segno della presenza di animali. Non c'erano nidi, non c'erano escrementi e non c'erano buchi nel legno o nei muri, nemmeno vicino alla finestra. Ha aggiunto che lo spazio era molto pulito e asciutto, e che il tetto sembrava in buone condizioni in ogni punto. Prima di andare via ha lasciato alcune trappole per sicurezza, ma ha detto che probabilmente sarebbero rimaste vuote. Infatti, nelle due settimane seguenti le trappole sono rimaste vuote, mentre il rumore continuava a ripetersi quasi tutte le sere.
Un venerdì sera, mentre usciva dal palazzo, Elena si è fermata a parlare con Aldo, che stava chiudendo la porta principale. Gli ha raccontato la storia del rumore, del tecnico e delle trappole vuote, e gli ha chiesto se avesse mai sentito qualcosa di simile. Aldo ha ascoltato tutta la storia senza interrompere, come faceva sempre quando qualcuno gli parlava di un problema del palazzo. Poi ha detto che in trent'anni non aveva mai sentito rumori strani al terzo piano, né di giorno né di sera. Ha ricordato però che, all'inizio di ottobre, il comune aveva fatto sostituire tutto l'impianto di riscaldamento del palazzo. I lavori erano durati tre settimane, e durante quel periodo l'edificio era stato pieno di operai, di attrezzi e di polvere. Gli operai avevano installato una caldaia nuova e anche dei tubi nuovi, che passavano proprio sopra il soffitto dell'ultimo piano. "Forse non è un caso: il rumore è cominciato poche settimane dopo i lavori sul riscaldamento", ha detto Aldo con calma. Ha aggiunto che, se voleva, poteva darle gli orari del nuovo impianto, perché erano programmati con un timer automatico. Aldo ha spiegato che il timer accendeva e spegneva la caldaia ogni giorno alla stessa ora, senza bisogno di nessun intervento. Elena non aveva mai pensato al riscaldamento, perché i termosifoni della sala funzionavano bene e non facevano nessun rumore. Inoltre, i tubi non si vedevano, perché erano nascosti nello spazio sotto il tetto e nei muri spessi del palazzo. Tornando a casa, però, ha riflettuto sulle parole di Aldo e ha deciso di raccogliere informazioni in modo più preciso.
Dal lunedì seguente Elena ha preparato una tabella semplice, con una colonna per il giorno, una per l'ora e una per la durata. Aveva scelto di usare una tabella perché sapeva, dal suo lavoro in archivio, che i dati scritti sono più affidabili della memoria. Ogni volta che sentiva il rumore, guardava l'orologio e scriveva il momento esatto in cui il suono cominciava e finiva. Ha chiesto anche ad Aldo una copia degli orari del riscaldamento, che erano scritti su un foglio appeso vicino alla caldaia. Il foglio indicava che l'impianto si accendeva alle sei del mattino e si spegneva alle sei e mezza di sera, dal lunedì al venerdì. Nel fine settimana, invece, il riscaldamento restava spento, perché nel palazzo non lavorava nessuno e gli uffici erano chiusi. Nelle prime sere i numeri non le dicevano molto, ma dopo alcuni giorni Elena ha cominciato a vedere una certa regolarità. Per due settimane ha scritto i dati ogni giorno, anche nelle sere in cui il rumore era molto debole e difficile da sentire. Anche Paolo l'ha aiutata, perché nei giorni in cui Elena usciva prima, restava lui in sala e annotava gli orari nella tabella. Il rumore non arrivava mai nel mezzo della giornata, quando la sala era calda e il riscaldamento funzionava senza interruzioni. Una domenica è passata in archivio per prendere un libro, e ha notato che la sala era silenziosa per tutto il pomeriggio. Alla fine di due settimane ha messo la sua tabella accanto al foglio degli orari, e ha confrontato le due serie di numeri.
Il risultato era molto chiaro: il rumore cominciava sempre pochi minuti dopo lo spegnimento del riscaldamento, verso le sei e quaranta. In tutte le sere registrate, la differenza tra lo spegnimento dell'impianto e l'inizio dei colpi era sempre di circa dieci minuti. Al mattino, invece, i colpi arrivavano poco dopo l'accensione, mentre l'acqua calda cominciava a scorrere di nuovo nei tubi. Nei fine settimana, quando l'impianto restava spento, il rumore non si sentiva mai, né al mattino né alla sera. Elena ha mostrato la tabella ad Aldo, che l'ha letta con attenzione e ha confrontato ogni ora con il foglio della caldaia. Aldo ha detto che i numeri non lasciavano dubbi, e ha telefonato subito alla ditta che aveva installato il nuovo impianto. Qualche giorno dopo è arrivato un idraulico, che è salito nello spazio sotto il tetto insieme ad Aldo, con una torcia e alcuni attrezzi. Prima di salire, l'idraulico aveva guardato la tabella di Elena, e aveva detto che quei dati gli avrebbero fatto risparmiare molto tempo. Dopo pochi minuti ha trovato il punto: un tubo di metallo appoggiato direttamente su una vecchia trave di legno, vicino alla finestra. L'idraulico ha chiesto a Elena di restare nella sala grande, e poi ha battuto piano sul tubo con un piccolo attrezzo. Elena ha sentito lo stesso suono secco delle settimane precedenti, e ha confermato che veniva esattamente da quel punto del soffitto. L'idraulico ha spiegato che il metallo si allunga un poco quando diventa caldo e si accorcia di nuovo quando si raffredda. Ogni volta che la temperatura cambiava, il tubo scivolava sulla trave a piccoli scatti, e ogni scatto produceva un colpo secco. "È un fenomeno normale, che succede spesso quando i tubi nuovi vengono montati troppo vicino al legno", ha detto l'idraulico.
Per risolvere il problema, l'idraulico ha sollevato leggermente il tubo e ha inserito tra il metallo e la trave alcuni supporti di gomma. La gomma permetteva al tubo di muoversi liberamente, senza strisciare sul legno e quindi senza produrre rumore durante i cambi di temperatura. Secondo l'idraulico, quei supporti erano economici e semplici da montare, e potevano durare molti anni senza bisogno di manutenzione. Ha controllato anche gli altri tubi sopra il terzo piano, ma erano tutti montati correttamente e lontani dalle travi. L'idraulico ha spiegato che era meglio controllare tutto subito, perché lo stesso problema poteva comparire anche in altri punti. Prima di scendere, Aldo ha scritto la data dell'intervento nel quaderno dove registrava tutte le riparazioni del palazzo. Quella sera Elena è rimasta in archivio fino alle sette, come al solito, e ha ascoltato con attenzione il silenzio della sala. Alle sei e mezza il riscaldamento si è spento, ma dal soffitto non è arrivato nessun colpo, né in quel momento né più tardi. Il giorno dopo ha raccontato a Paolo come erano andate le cose, e insieme hanno tolto le trappole che erano rimaste nel corridoio. Paolo ha osservato che era meglio così, perché i documenti più antichi erano delicati e difficili da sostituire. Elena ha conservato la tabella in una cartella dell'archivio, insieme alla documentazione dei lavori sul riscaldamento del palazzo. Pensava che un giorno qualcun altro avrebbe potuto trovare utile quella informazione, magari durante altri lavori sull'edificio. Poi è tornata ai suoi registri antichi, e le settimane successive sono passate tranquille, tra le richieste dei cittadini e i documenti da catalogare.""")


text = text[:500]  # limit to 4096 characters for demo
target_voice = "im_nicola"     # timbre + segments (native Italian voice)
donor_voice  = "hf_alpha"      # prosody source, from another language (e.g. American English)

ipa_chunks = text_to_ipa_chunks(text)
for ps in ipa_chunks:
    print(ps)

ˈɛːlena markˈetːɪ lavorˈava da sˌɛi ˈannɪ nellarkˈivio komʊnˈale dɪ pontevˈɛrde, ˌʊna pˈikːola ʧitːˈa kostrʊˈita lˈuŋɡo ʊn fjˈume nˈella pjanˈura del nˈɔrd.
larkˈivio okːʊpˈava il tˈɛrʦo pjˈano dɪ ʊn vˈɛkːio palˈaʦːo dɪ matːˈonɪ rˈoSɪ, a pˈɔkɪ pˈaSɪ dˌalla pjˈaʦːa prinʧipˈale e dal mʊniʧˈipio.
il palˈaʦːo ˌɛra stˈato kostrʊˈito ˌalla fˈine dellotːoʧˈɛnto, e in paSˈato avˌeva ospitˈato ˌʊna skʊˈɔla e pˈɔi alkˌʊnɪ ʊffˈiʧɪ del tribʊnˈale.
la sˈala pjʊ ɡrˈande avˌeva soffˈitːɪ ˈaltɪ, skaffˈalɪ dɪ lˈeɲɲo skˈuro e dˈue lˈuŋɡe finˈɛstre ke ɡwˈarda


## Native: the target voice with its own prosody
This uses the same pipeline with nothing transferred, so both conditions go through identical processing.

In [8]:
audio_nat, segs_nat, words_nat, alt_nat, pros_nat = synth_prosody(
    ipa_chunks, target_voice, donor_voice, pitch=False, energy=False, duration=False)

display(HTML(audiovisualize_prosody(audio_nat, segs_nat, pros_nat, seg_words=words_nat, text=text,
                                    seg_altered=alt_nat, title=f"Native: {target_voice}",
                                    out_html="native.html")))

Writing interactive HTML to: c:\Users\gmoha\OneDrive\文档\GitHub\PhD-HUM-AI-N-ACCENT\artificial_stim\output_online\native.html


## Accented: the target voice with the donor's prosody
Set the flags to choose what is transferred.

In [9]:
audio_acc, segs_acc, words_acc, alt_acc, pros_acc = synth_prosody(
    ipa_chunks, target_voice, donor_voice,
    pitch=True, energy=True, duration=True,
    f0_mode="meanstd", energy_mode="meanstd", dur_mode="rate", norm_scope="text",
    dur_scope="sentence")    # "sentence": every sentence as long as the native one; "text": only the total

display(HTML(audiovisualize_prosody(audio_acc, segs_acc, pros_acc, seg_words=words_acc, text=text,
                                    seg_altered=alt_acc,
                                    title=f"Accented: {target_voice} with {donor_voice}'s prosody",
                                    out_html="accented_prosody.html")))

Writing interactive HTML to: c:\Users\gmoha\OneDrive\文档\GitHub\PhD-HUM-AI-N-ACCENT\artificial_stim\output_online\accented_prosody.html


## Save the audio

In [ ]:
os.makedirs("output_online", exist_ok=True)
sf.write("output_online/native.wav", audio_nat, SR)
sf.write("output_online/accented_prosody.wav", audio_acc, SR)
print(f"native   {len(audio_nat)/SR:7.3f} s")
print(f"accented {len(audio_acc)/SR:7.3f} s")

# per-sentence check (frames of 25 ms)
for k, ((a_n, i_n), (a_a, i_a)) in enumerate(zip(pros_nat["chunks"], pros_acc["chunks"])):
    print(f"sentence {k}: native {sum(i_n['dur']):4d} fr / {len(a_n)/SR:6.3f} s   "
          f"accented {sum(i_a['dur']):4d} fr / {len(a_a)/SR:6.3f} s")

## Several conditions in one go
One player and one wav per condition, all from the same text, e.g. for a listening test.

In [ ]:
conditions = {
    "pitch":                 dict(pitch=True,  energy=False, duration=False),
    "pitch+energy":          dict(pitch=True,  energy=True,  duration=False),
    "duration":              dict(pitch=False, energy=False, duration=True),
    "pitch+energy+duration": dict(pitch=True,  energy=True,  duration=True),
}

results = {}
for name, flags in conditions.items():
    a, s, w, alt, p = synth_prosody(ipa_chunks, target_voice, donor_voice, **flags)
    results[name] = (a, s, w, alt, p)
    fname = name.replace("+", "_")
    sf.write(f"output_online/{fname}.wav", a, SR)
    display(HTML(audiovisualize_prosody(a, s, p, seg_words=w, text=text, seg_altered=alt,
                                        title=f"{name}: {target_voice} with {donor_voice}'s prosody",
                                        out_html=f"{fname}.html")))

## (Optional) Segmental + prosodic accent together
Apply your `manipulate()` rules to the IPA first, then transfer the prosody onto the manipulated text.
Altered phonemes are highlighted in the player as usual.
Both voices predict durations for the manipulated string itself, the equivalent of `dur_source="manipulated"`.

In [ ]:
rules = [{"kind": "substitute_prob", "old": "d", "choices": {"ʈ": 100}, "same_in_word": True},
        {"kind": "substitute_prob", "old": "k", "choices": {"χ": 100}, "same_in_word": True},
        {"kind": "substitute_prob", "old": "e", "choices": {"ø": 100}, "same_in_word": True}]    # <- put your manipulation rules here

if rules:
    manip_chunks, masks, stats = manipulate(ipa_chunks, rules, return_mask=True, return_stats=True, seed=1)
    a, s, w, alt, p = synth_prosody(manip_chunks, target_voice, donor_voice,
                                    pitch=True, energy=True, duration=True, masks=masks)
    sf.write("output_online/segmental_prosody.wav", a, SR)
    display(HTML(audiovisualize_prosody(a, s, p, seg_words=w, text=text, seg_altered=alt,
                                        rules=rules, stats=stats,
                                        title=f"Segmental + prosodic accent ({donor_voice})",
                                        out_html="segmental_prosody.html")))

Writing interactive HTML to: c:\Users\gmoha\OneDrive\文档\GitHub\PhD-HUM-AI-N-ACCENT\artificial_stim\output_online\segmental_prosody.html
